# Bi-GRU với PhoW2V 300dims trên ViCTSD

So sánh cùng kiến trúc khi train trên nhãn gốc và nhãn AI re-annotated.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/lechihoang/multi-agent-annotation.git'
PROJECT_DIR = Path('/kaggle/working/multi-agent-annotation')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR / 'notebooks')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR), 'tensorflow', 'gensim'], check=True)

sys.path.insert(0, str(PROJECT_DIR))
root = PROJECT_DIR
print(f'Project root: {PROJECT_DIR}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.experiment_utils import (
    build_embedding_matrix,
    compare_prediction_disagreements,
    load_victsd_splits,
    prepare_keras_text_data,
    train_predict_keras_pair,
)

sns.set_theme(style='whitegrid')

MAX_WORDS = 10000
MAX_LEN = 100
EMBEDDING_DIM = 300

In [ ]:
splits = load_victsd_splits(root / 'data')
keras_data = prepare_keras_text_data(splits, max_words=MAX_WORDS, max_len=MAX_LEN)
embedding_matrix = build_embedding_matrix(
    keras_data['tokenizer'],
    max_words=MAX_WORDS,
    embedding_dim=EMBEDDING_DIM,
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, GRU
from tensorflow.keras.models import Sequential

tf.random.set_seed(42)

def build_model():
    model = Sequential([
        Embedding(MAX_WORDS, EMBEDDING_DIM, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True),
        Bidirectional(GRU(64, dropout=0.2, recurrent_dropout=0.2)),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
model_orig, model_new, y_pred_orig, y_pred_new, comparison = train_predict_keras_pair(
    build_model=build_model,
    X_train_original=keras_data['X_train_original'],
    y_train_original=keras_data['y_train_original'],
    X_train_reannotated=keras_data['X_train_reannotated'],
    y_train_reannotated=keras_data['y_train_reannotated'],
    X_test=keras_data['X_test'],
    y_test=keras_data['y_test'],
    model_name='Bi-GRU (PhoW2V 300dims)',
    epochs=5,
    batch_size=32,
)

In [ ]:
diff_df = compare_prediction_disagreements(
    keras_data['test_texts'],
    keras_data['y_test'],
    y_pred_orig,
    y_pred_new,
)
diff_df.head(10)